In [1]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# The Usual Suspects

There has been a burglary at a jewelry store.  The [usual suspects](https://en.wikipedia.org/wiki/The_Usual_Suspects) have been arrested.  These are
<ul>
<li>Aaron,</li>
<li>Bernard, and</li>
<li>Caine.</li>
</ul>
Furthermore, the following facts have been established:
<ol>
<li>It is known that at least one of these suspects is indeed guilty.</li>
<li>If Aaron is guilty, he has exactly one accomplice.</li>
<li>If Bernard is innocent, then Caine is inncocent, too.</li>
<li>If exactly two of the suspects are guilty, then Caine is one of them.</li>
<li>If Caine is innocent, then Aaron is guilty.</li>
</ol>
It is our task to identify those suspects that are guilty.

Our first task is to define the set of propositional variables:
$$ \mathcal{P} := \{ \texttt{a}, \texttt{b}, \texttt{c} \} $$
The interpretation is that 
<ul>
<li>$\texttt{a}$ is true iff Aaron is guilty,</li> 
<li>$\texttt{b}$ is true iff Bernard is guilty, and</li>
<li>$\texttt{c}$ is true iff Caine is guilty.  </li>
</ul>

In [2]:
const P = new Set(['a', 'b', 'c']);

Our next task is to translate the facts given above into formulas from propositional logic. 

The statement "It is known that at least one of these suspects is indeed guilty." is translated as follows:
$$ 
\texttt{a} \vee \texttt{b} \vee \texttt{c}. 
$$ 

In [3]:
const f1 = 'a ∨ b ∨ c';

The statement "If Aaron is guilty, he has exactly one accomplice." is harder to translate into propositional logic. The idea is to split this statement into two statements:
* If Aaron is guilty, he has at least one accomplice.</li>
* If Aaron is guilty, he has at most  one accomplice.</li>

These statements can now be translated into the following formulas:

In [4]:
const f2 = 'a → b ∨ c';

In [5]:
const f3 = 'a → ¬(b ∧ c)';

The statement "If Bernard is innocent, then Caine is inncocent, too." is a simple implication:

In [6]:
const f4 = '¬b → ¬c';

The statement "If exactly two of the suspects are guilty, then Caine is one of them." is best translated into propositional logic by asking how this statement could be made false.
Obviously, this statement is false if two suspects are guilty, but Caine is innocent.
But this is only possible if Caine is innocent and Aaron and Bernard are true.  Hence we can translate this statement as follows:

In [7]:
const f5 = '¬(¬c ∧ a ∧ b)';

The statement "If Caine is innocent, then Aaron is guilty." is an implication:

In [8]:
const f6 = '¬c → a';

We define the set `Fs` of all formulas:

In [9]:
const Fs = new Set([f1, f2, f3, f4, f5, f6]);

We need to transform the strings <tt>f1</tt> to <tt>f6</tt> into nested tuples representing formulas.  To this end we import a parser for propositional formulas.

In [10]:
import { LogicParser } from './PropositionalLogicParser'

In [11]:
type Formula = string | [string, ...Formula[]];

In [12]:
function parse(s: string): Formula {
    const parser = new LogicParser(s);
    return parser.parse();
}

Next, we transform all formulas into nested tuples:

In [13]:
const Gs = Array.from(Fs).map(f => parse(f));
console.dir(Gs, {depth: null})

[
  [ '∨', [ '∨', 'a', 'b' ], 'c' ],
  [ '→', 'a', [ '∨', 'b', 'c' ] ],
  [ '→', 'a', [ '¬', [ '∧', 'b', 'c' ] ] ],
  [ '→', [ '¬', 'b' ], [ '¬', 'c' ] ],
  [
    '¬',
    [ '∧', [ '∧', [ '¬', 'c' ], 'a' ], 'b' ]
  ],
  [ '→', [ '¬', 'c' ], 'a' ]
]


We are looking for a variable assignment $\mathcal{I}$ that satisfies all formulas in the set <tt>Fs</tt>.  As variable assignments are represented as subsets of the set $\mathcal{P}$ of propositional variables, we can just iterate over all subsets of $\mathcal{P}$.

The function `allSubsets(M)` takes a set `M` as its input and returns the list of all subsets of `M`.
The idea behind the definition of `allSubsets` is as follows:
1. Let `x` be any element from the set `M`. 
2. Then there are two kinds of subsets of `M`:
   * Those subsets $A \subseteq M$ that do not contain `x`.
   * Those subsets $B \subseteq M$ that do contain `x`.
3. The set $\mathcal{L}$ of those subsets `A` of `M` that do not contain `x` can be calculated recursively:
   $$ \mathcal{L} = \texttt{allSubsets}(M - \{x\}) $$
4. Adding `x` to the subsets in $\mathcal{L}$ yields all those subsets of $M$ that do contain `x`. 

In [14]:
function allSubsets<T>(M: Set<T>): Set<T>[] {
    if (M.size === 0) {
        return [new Set<T>()];
      }
    const x = M.values().next().value;
    M.delete(x);
    const L = allSubsets(M);
    const withX = L.map(A => new Set([...A, x]));
    return [...L, ...withX];
}

In [15]:
allSubsets(new Set([1, 2, 3]));

[
  Set(0) {},
  Set(1) { 3 },
  Set(1) { 2 },
  Set(2) { 3, 2 },
  Set(1) { 1 },
  Set(2) { 3, 1 },
  Set(2) { 2, 1 },
  Set(3) { 3, 2, 1 }
]


The function $\texttt{evaluate}(F, I)$ takes a propositional formula $F$ and a propositional variable assignment $I$ and evaluates $F$ using the assignment $I$.  We have discussed the details of this function previously.

In [16]:
function evaluate(F: Formula, I: Set<string>): boolean {
    if (typeof F === 'string')
        { return I.has(F);}
    if (F[0] === '⊤')   return true;
    if (F[0] === '⊥')   return false;
    if (F[0] === '¬') { return !evaluate(F[1], I);}
    if (F[0] === '∧') { return  evaluate(F[1], I) &&   evaluate(F[2], I);}
    if (F[0] === '∨') { return  evaluate(F[1], I) ||   evaluate(F[2], I);}
    if (F[0] === '→') { return !evaluate(F[1], I) ||   evaluate(F[2], I);}
    if (F[0] === '↔') { return  evaluate(F[1], I) ===  evaluate(F[2], I);}
    return false;
}

The function `allTrue(Fs, I)` takes a set of propositional formula  `Fs`
and a propositional variable assignment `I`.  It returns `true` only if all formulas from `Fs` are 
`true` given the variable assignment `I`.

In [17]:
function allTrue(Gs: Formula[], I: Set<string>): boolean {
    return Gs.every(f => evaluate(f, I));
}

Next, we compute the set of all variable assignments that render all formulas true:

In [18]:
const validAssignments = allSubsets(P).filter(I => allTrue(Gs, I));
console.log(validAssignments);

[ Set(2) { 'c', 'b' } ]


UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

...

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

It turns our that there is just one propositional variable assignment that satisfies all formulas from the set <tt>Fs</tt>.  Therefore, the problem has a unique solution: Bernard and Caine are guilty, while Aaron is innocent.